# 63 — Référence externe : LightRAG

Exécute **LightRAG 1.5.7** de bout en bout sur le même corpus de 475 résumés. LightRAG construit lui-même son graphe de connaissances, puis répond aux 20 questions ; son rappel est mesuré exactement comme celui des autres baselines (documents distincts, dans l'ordre des extraits qu'il renvoie).

Pour que la comparaison porte sur la méthode et non sur les moyens :
- même modèle de langue que notre pipeline : **qwen2.5:7b** via Ollama, température 0 ;
- même encodeur que la baseline vectorielle : **S-PubMedBert-MS-MARCO** ;
- un résumé forme un seul fragment (1 200 tokens), comme dans notre propre extraction.

Les autres paramètres de LightRAG restent à leurs valeurs par défaut. Le reclassement est désactivé, faute de modèle de reclassement. Deux modes de recherche sont évalués : `mix` (mode par défaut : graphe et recherche vectorielle sur les fragments) et `hybrid` (graphe seul).

**Deux temps.** Cellule 5, `PILOTE = True` : 10 documents, pour mesurer la durée et la qualité de l'extraction avant de s'engager. Puis `PILOTE = False` : les 475 documents et l'évaluation.

**Reprise après coupure.** Le dossier de travail est sur Google Drive. Si la session s'interrompt, relancer toutes les cellules : les documents déjà traités ne sont pas refaits, ceux restés en suspens sont repris.

**Entrées** : `corpus.json`, `jeu_de_test.json`, `courbe_rappel.json`
**Sortie** : `baseline_lightrag.json`, à placer dans `data/processed/`

## Cellule 1 — Ollama : installation et serveur

In [1]:
import subprocess, time, requests, os

def ollama_repond():
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not os.path.exists("/usr/local/bin/ollama"):
    print("Installation des prérequis et d'Ollama (~2 min)...")
    subprocess.run("apt-get install -y -qq zstd pciutils lshw", shell=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

if not ollama_repond():
    print("Démarrage du serveur (4 requêtes en parallèle)...")
    subprocess.Popen(["ollama", "serve"],
                     env={**os.environ, "OLLAMA_NUM_PARALLEL": "4"},
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    for _ in range(30):
        if ollama_repond():
            break
        time.sleep(1)

print("Ollama :", "actif" if ollama_repond() else "ARRÊTÉ")

Installation des prérequis et d'Ollama (~2 min)...
Démarrage du serveur (4 requêtes en parallèle)...
Ollama : actif


## Cellule 2 — Le modèle

In [2]:
!ollama pull qwen2.5:7b
!ollama list


NAME          ID              SIZE      MODIFIED               
qwen2.5:7b    845dbda0ea48    4.7 GB    Less than a second ago    


## Cellule 3 — LightRAG, version figée

In [3]:
!pip install -q lightrag-hku==1.5.7 sentence-transformers ollama
import lightrag
print("LightRAG", lightrag.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 8.9 MB/s eta 0:00:00
LightRAG 1.5.7


## Cellule 4 — Drive et fichiers

In [4]:
from google.colab import drive, files
import json, os, shutil

drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/stage-graphrag"
os.makedirs(DOSSIER, exist_ok=True)

NECESSAIRES = ["corpus.json", "jeu_de_test.json", "courbe_rappel.json"]
manquants = [n for n in NECESSAIRES if not os.path.exists(f"{DOSSIER}/{n}")]
if manquants:
    print("À téléverser :", manquants)
    for nom in files.upload():
        shutil.copy(nom, f"{DOSSIER}/{nom}")
for nom in NECESSAIRES:
    print(f"{nom:<22}", "présent" if os.path.exists(f"{DOSSIER}/{nom}") else "ABSENT")

Mounted at /content/drive
À téléverser : ['courbe_rappel.json']


Saving corpus.json to corpus.json
Saving courbe_rappel.json to courbe_rappel.json
Saving jeu_de_test.json to jeu_de_test.json
corpus.json            présent
jeu_de_test.json       présent
courbe_rappel.json     présent


## Cellule 5 — Configuration

In [5]:
import random
import lightrag
from sentence_transformers import SentenceTransformer
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete
from lightrag.utils import EmbeddingFunc

# ------------------------------------------------------------------
PILOTE = True       # True : 10 documents ; False : les 475
GLANAGE = 1         # passes d'extraction supplémentaires (défaut LightRAG : 1)
# ------------------------------------------------------------------

TRAVAIL = f"{DOSSIER}/lightrag_{'pilote' if PILOTE else 'complet'}"
os.makedirs(TRAVAIL, exist_ok=True)

with open(f"{DOSSIER}/corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)
with open(f"{DOSSIER}/jeu_de_test.json", encoding="utf-8") as f:
    questions = json.load(f)
par_pmid = {a["pmid"]: a for a in corpus}

# Pilote : les documents attendus de quatre questions, plus six tirés au
# hasard (graine 42), pour mesurer la durée et la qualité de l'extraction.
QUESTIONS_PILOTE = ["SH01", "SH07", "SH09", "MH02"]
if PILOTE:
    choisis = [p for q in questions if q["id"] in QUESTIONS_PILOTE for p in q["pmids"]]
    reste = sorted(set(par_pmid) - set(choisis))
    random.seed(42)
    choisis += random.sample(reste, 10 - len(choisis))
    DOCUMENTS = [par_pmid[p] for p in choisis]
else:
    DOCUMENTS = corpus
print(f"Mode {'PILOTE' if PILOTE else 'COMPLET'} : {len(DOCUMENTS)} documents")
print("Dossier de travail :", TRAVAIL)

# Encodeur : le même modèle que la baseline vectorielle
encodeur = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO", device="cuda")

async def encoder(textes, **kwargs):
    return encodeur.encode(list(textes), normalize_embeddings=True, convert_to_numpy=True)

rag = LightRAG(
    working_dir=TRAVAIL,
    llm_model_func=ollama_model_complete,
    llm_model_name="qwen2.5:7b",
    llm_model_kwargs={"host": "http://127.0.0.1:11434", "timeout": 900,
                      "options": {"num_ctx": 16384, "temperature": 0}},
    llm_model_max_async=4,
    embedding_func=EmbeddingFunc(embedding_dim=768, func=encoder),
    chunk_token_size=1200,
    entity_extract_max_gleaning=GLANAGE,
)
await rag.initialize_storages()

# Compte les erreurs de format signalées par LightRAG pendant l'extraction :
# c'est le premier indicateur de la capacité du modèle à suivre son format.
import logging
class CompteurFormat(logging.Handler):
    def __init__(self):
        super().__init__(logging.WARNING)
        self.n, self.exemples = 0, []
    def emit(self, r):
        m = r.getMessage()
        if "format error" in m:
            self.n += 1
            if len(self.exemples) < 3:
                self.exemples.append(m[:150])
compteur = CompteurFormat()
logging.getLogger("lightrag").addHandler(compteur)
print("LightRAG prêt")

Mode PILOTE : 10 documents
Dossier de travail : /content/drive/MyDrive/stage-graphrag/lightrag_pilote


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO: [] Created new empty graph file: /content/drive/MyDrive/stage-graphrag/lightrag_pilote/graph_chunk_entity_relation.graphml
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: [] Process 1022 KV load full_docs with 0 records
INFO: [] Process 1022 KV load text_chunks with 0 records
INFO: [] Process 1022 KV load full_entities with 0 records
INFO: [] Process 1022 KV load full_relations with 0 records
INFO: [] Process 1022 KV load entity_chunks with 0 records
INFO: [] Process 1022 KV load relation_chunks with 0 records
INFO: [] Process 1022 KV load llm_response_cache with 0 records
INFO: [] Process 1022 doc status load doc_status with 0 records


LightRAG prêt


## Cellule 6 — Extraction (la longue)

In [6]:
import time
from lightrag.base import DocStatus

# Reprise : un document déjà connu de LightRAG n'est jamais réinséré.
# Ceux qu'une coupure a laissés en suspens sont repris par sa propre file.
connus = await rag.doc_status.get_docs_by_statuses(list(DocStatus))
deja = await rag.doc_status.get_docs_by_statuses([DocStatus.PROCESSED])
nouveaux = [a for a in DOCUMENTS if a["pmid"] not in connus]
en_suspens = len(connus) - len(deja)
print(f"Déjà traités : {len(deja)}   en suspens : {en_suspens}   nouveaux : {len(nouveaux)}")

debut = time.time()
if nouveaux:
    await rag.ainsert([a["titre"] + "\n\n" + a["resume"] for a in nouveaux],
                      ids=[a["pmid"] for a in nouveaux],
                      file_paths=[a["pmid"] for a in nouveaux])
elif en_suspens:
    await rag.apipeline_process_enqueue_documents()
duree = time.time() - debut

traites = await rag.doc_status.get_docs_by_statuses([DocStatus.PROCESSED])
echoues = await rag.doc_status.get_docs_by_statuses([DocStatus.FAILED])
faits = len(traites) - len(deja)
print(f"\nDurée de cette exécution : {duree / 60:.1f} min pour {faits} documents")
if faits:
    par_doc = duree / faits
    print(f"  soit {par_doc:.0f} s par document ; "
          f"projection pour 475 documents : {475 * par_doc / 3600:.1f} h")
print(f"Traités : {len(traites)} / {len(DOCUMENTS)}   échoués : {len(echoues)}")
for pm, st in list(echoues.items())[:5]:
    print("  ÉCHEC", pm, str(getattr(st, "error_msg", st))[:160])
print(f"Erreurs de format signalées par LightRAG : {compteur.n}")
for e in compteur.exemples:
    print("   ", e)

INFO: Processing 10 document(s)
INFO: Parsing (native): 33309519
INFO: Parsing (native): 27628007
INFO: Parsing (native): 26432838
INFO: Parsing (native): 29649585
INFO: Parsing (native): 17243942
INFO: Parsing (native): 36350591
INFO: Parsing (native): 15585743
INFO: Parsing (native): 11175438
INFO: Parsing (native): 38851804
INFO: Parsing (native): 22302382


Déjà traités : 0   en suspens : 0   nouveaux : 10


INFO: Extracting stage 1/10: 33309519
INFO: Processing d-id: 33309519
INFO: Extracting stage 2/10: 27628007
INFO: Processing d-id: 27628007
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: 33309519
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: 27628007
INFO: Extracting stage 3/10: 26432838
INFO: Processing d-id: 26432838
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: 26432838
INFO: extract LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: default:extract:b59581d3f3ec9fb59249e79a02cf0b30
INFO:  == LLM cache == saving: default:extract:2acfd5bde85fb1e68fbcace4d53a9687
INFO: Chunk 1 of 1 extracted 18 Ent + 14 Rel 33309519-chunk-000
INFO: Merging stage 1/10: 33309519
INFO: Phase 0: Writing recovery indexes for 33309519: 18 candidate entities, 14 candidate relations
INFO: Phase 1: Processing 18 entities from 33309519 (async: 8)
INFO: 


Durée de cette exécution : 17.5 min pour 5 documents
  soit 210 s par document ; projection pour 475 documents : 27.7 h
Traités : 5 / 10   échoués : 5
  ÉCHEC 27628007 C[1/1]: 27628007-chunk-000: extract LLM func: Worker execution timeout after 480s
  ÉCHEC 15585743 C[1/1]: 15585743-chunk-000: extract LLM func: Worker execution timeout after 480s
  ÉCHEC 17243942 C[1/1]: 17243942-chunk-000: extract LLM func: Worker execution timeout after 480s
  ÉCHEC 26432838 C[1/1]: 26432838-chunk-000: extract LLM func: Worker execution timeout after 480s
  ÉCHEC 29649585 C[1/1]: 29649585-chunk-000: extract LLM func: Worker execution timeout after 480s
Erreurs de format signalées par LightRAG : 3
    36350591-chunk-000: LLM output format error; found 6/5 fields on RELATION `Mindfulness-Based Stress Reduction`~`vs`
    36350591-chunk-000: LLM output format error; found 6/5 fields on RELATION `Mindfulness-Based Stress Reduction`~`noninferior`
    36350591-chunk-000: LLM output format error; found 6/5 

## Cellule 7 — Qualité de l'extraction

In [7]:
import networkx as nx

graphe = nx.read_graphml(f"{TRAVAIL}/graph_chunk_entity_relation.graphml")
print(f"Graphe LightRAG : {graphe.number_of_nodes()} entités, {graphe.number_of_edges()} relations")

par_doc = {a["pmid"]: [0, 0] for a in DOCUMENTS}
for _, d in graphe.nodes(data=True):
    for pm in str(d.get("file_path", "")).split("<SEP>"):
        if pm in par_doc:
            par_doc[pm][0] += 1
for _, _, d in graphe.edges(data=True):
    for pm in str(d.get("file_path", "")).split("<SEP>"):
        if pm in par_doc:
            par_doc[pm][1] += 1
vides = [pm for pm, (e, r) in par_doc.items() if e == 0]
print(f"Documents sans aucune entité extraite : {len(vides)}"
      + (f"  -> {vides[:10]}" if vides else ""))
ent = sorted(e for e, _ in par_doc.values())
rel = sorted(r for _, r in par_doc.values())
print(f"Par document : entités médiane {ent[len(ent) // 2]} (min {ent[0]}, max {ent[-1]}), "
      f"relations médiane {rel[len(rel) // 2]} (min {rel[0]}, max {rel[-1]})")

print("\nÉchantillon de relations extraites :")
random.seed(42)
aretes = list(graphe.edges(data=True))
for s, t, d in random.sample(aretes, min(12, len(aretes))):
    print(f"  {s} -> {t}")
    print(f"      mots-clés : {str(d.get('keywords', ''))[:90]}")
    print(f"      source    : {d.get('file_path', '')}")

Graphe LightRAG : 88 entités, 45 relations
Documents sans aucune entité extraite : 5  -> ['27628007', '26432838', '29649585', '17243942', '15585743']
Par document : entités médiane 13 (min 0, max 26), relations médiane 2 (min 0, max 19)

Échantillon de relations extraites :
  VEGF Trap -> also known as
      mots-clés : Aflibercept
      source    : 22302382
  P2Y12Receptor -> CrystallographyOfHumanP2Y12Receptor
      mots-clés : MethodUsedForStructuralInformation
      source    : 33309519
  P2Y1Receptor -> CrystallographyOfHumanP2Y1Receptor
      mots-clés : MethodUsedForStructuralInformation
      source    : 33309519
  Escitalopram -> dosage
      mots-clés : flexibly dosed from 10 to 20 mg
      source    : 36350591
  Mindfulness-Based Stress Reduction -> course
      mots-clés : 8 weeks
      source    : 36350591
  Mindfulness-Based Stress Reduction -> decreases
      mots-clés : anxiety levels
      source    : 36350591
  P2Y13Receptor -> P2YReceptors
      mots-clés : Subtype
 

## Cellule 8 — Rappel de LightRAG

In [8]:
BUDGETS = (1, 3, 5, 10, 20)
MODES = ("mix", "hybrid")
evaluees = [q for q in questions if (not PILOTE) or q["id"] in QUESTIONS_PILOTE]

CLASSEMENTS = {m: {} for m in MODES}
for q in evaluees:
    for mode in MODES:
        r = await rag.aquery_data(q["question"], param=QueryParam(mode=mode, enable_rerank=False))
        vus = []
        for c in r.get("data", {}).get("chunks", []):
            pm = str(c.get("file_path", ""))
            if pm and pm not in vus:
                vus.append(pm)
        CLASSEMENTS[mode][q["id"]] = vus

def rappel(cl, K, categorie=None):
    qs = [q for q in evaluees if categorie is None or q["type"] == categorie]
    return sum(len(set(q["pmids"]) & set(cl[q["id"]][:K])) / len(q["pmids"]) for q in qs) / len(qs)

n_docs = {m: sorted(len(v) for v in CLASSEMENTS[m].values()) for m in MODES}
for m in MODES:
    print(f"Mode {m:<7} documents renvoyés par question : médiane {n_docs[m][len(n_docs[m]) // 2]}, "
          f"min {n_docs[m][0]}, max {n_docs[m][-1]}")

if PILOTE:
    print("\nContrôle du pilote (10 documents seulement : ce n'est pas une mesure)")
    for q in evaluees:
        print(f"  {q['id']}  attendus {q['pmids']}")
        for m in MODES:
            print(f"      {m:<7} renvoie {CLASSEMENTS[m][q['id']][:5]}")
else:
    with open(f"{DOSSIER}/courbe_rappel.json", encoding="utf-8") as f:
        GRAPHE = {l["id"]: l["graphe"] for l in json.load(f)["detail"]}
    colonnes = [(f"LightRAG {m}", CLASSEMENTS[m]) for m in MODES] + [("notre graphe", GRAPHE)]
    print(f"\n{'K':>13}" + "".join(f"{n:>17}" for n, _ in colonnes))
    for K in BUDGETS:
        print(f"{K:>13}" + "".join(f"{rappel(c, K):>17.2f}" for _, c in colonnes))
    print("\nRappel@5 par catégorie")
    for cat in ("single-hop", "pont", "intersection"):
        print(f"{cat:>13}" + "".join(f"{rappel(c, 5, cat):>17.2f}" for _, c in colonnes))

INFO: keyword LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: mix:keywords:745aa8925c575cec60762d1600480063
INFO: Query edges: signalling pathway, platelet receptor, cangrelor antagonizes (top_k:40, cosine:0.2)
INFO: Global query: 47 entites, 40 relations
INFO: Naive query: 10 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 47 entities, 40 relations, 10 vector chunks
INFO: After truncation: 47 entities, 40 relations
INFO: Selecting 5 from 5 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 40 relations
INFO: Round-robin merged chunks: 15 -> 10 (deduplicated 5)
INFO: Final context: 47 entities, 40 relations, 10 chunks
INFO: Final chunks S+F/O: E15/1 C1/2 E4/2 C1/3 E6/3 C1/4 E3/4 E20/5 C1/8 C1/10
INFO:  == LLM cache == saving: hybrid:keywords:2d47272b331ef6b13ed69828c0fbbda9
INFO: Query edges: signalling pathway, platelet receptor, cangrelor antagonizes

Mode mix     documents renvoyés par question : médiane 10, min 10, max 10
Mode hybrid  documents renvoyés par question : médiane 5, min 5, max 5

Contrôle du pilote (10 documents seulement : ce n'est pas une mesure)
  MH02  attendus ['33309519', '27628007']
      mix     renvoie ['33309519', '27628007', '11175438', '17243942', '22302382']
      hybrid  renvoie ['33309519', '11175438', '22302382', '38851804', '36350591']
  SH01  attendus ['26432838']
      mix     renvoie ['38851804', '26432838', '36350591', '29649585', '22302382']
      hybrid  renvoie ['38851804', '36350591', '22302382', '33309519', '11175438']
  SH07  attendus ['29649585']
      mix     renvoie ['33309519', '27628007', '11175438', '29649585', '22302382']
      hybrid  renvoie ['33309519', '11175438', '22302382', '38851804', '36350591']
  SH09  attendus ['17243942']
      mix     renvoie ['17243942', '33309519', '27628007', '22302382', '11175438']
      hybrid  renvoie ['33309519', '22302382', '11175438', '38851804', 

## Cellule 9 — Enregistrer (exécution complète seulement)

In [9]:
if PILOTE:
    print("Pilote : rien à enregistrer. Copie-moi les sorties des cellules 6, 7 et 8.")
else:
    resultat = {"lightrag": lightrag.__version__, "modele": "qwen2.5:7b",
                "glanage": GLANAGE, "documents_traites": len(traites),
                "documents_echoues": sorted(echoues), "classements": CLASSEMENTS}
    with open("baseline_lightrag.json", "w", encoding="utf-8") as f:
        json.dump(resultat, f, ensure_ascii=False, indent=1)
    shutil.copy("baseline_lightrag.json", f"{DOSSIER}/baseline_lightrag.json")
    files.download("baseline_lightrag.json")

Pilote : rien à enregistrer. Copie-moi les sorties des cellules 6, 7 et 8.
